# 03 Analytical Dataset Design

## Goal

The goal of this notebook is to design the final player-level analytical dataset for engagement and churn analysis.

The final dataset should contain one row per player and combine:
- player metadata,
- activity history,
- game ownership,
- social connections,
- derived engagement metrics.

## Step 1: Player Activity

We start by aggregating the history table to player level.

## Load Libraries

In [1]:
import os
from idlelib import query
from unittest import result

import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import create_engine, inspect, text

## Load Environment Variables

In [5]:
PROJECT_ROOT = Path("..")
PROCESSED_DATA_PATH = PROJECT_ROOT / "Data" / "processed"

load_dotenv(PROJECT_ROOT / ".env")

DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_SCHEMA = os.getenv("DB_SCHEMA", "steam")

## Create Database Connection

In [3]:
connection_string =(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)
engine = create_engine(connection_string)

## Run Queries

In [4]:
query = """
SELECT
    h.player_id,
    MIN(h.date_acquired) AS first_activity_date,
    MAX(h.date_acquired) AS last_activity_date,
    COUNT(*) AS total_achievements_unlocked,
    DATE_PART('day', MAX(h.date_acquired) - MIN(h.date_acquired)) AS active_days
FROM steam.history h
WHERE h.date_acquired IS NOT NULL
GROUP BY h.player_id
LIMIT 100;
"""

player_activity_sample = pd.read_sql(query, engine)

player_activity_sample.head()

,player_id,first_activity_date,last_activity_date,total_achievements_unlocked,active_days
0,76561197960270682,2017-11-06 06:56:29,2024-09-14 22:21:34,114477,2504.0
1,76561197960272112,1970-01-01 05:00:00,2024-12-03 03:40:39,6191,20059.0
2,76561197960272169,1970-01-01 05:00:00,2024-12-10 04:01:32,5624,20066.0
3,76561197960273069,1970-01-01 05:00:00,2024-12-09 06:29:39,1577,20066.0
4,76561197960273410,1970-01-01 05:00:00,2023-12-20 20:44:03,275,19711.0


In [9]:
player_activity_sample.to_csv(
    PROCESSED_DATA_PATH / "player_activity_sample.csv",
    index=False
)

In [7]:
query = """
SELECT
    COUNT(DISTINCT player_id) AS players_with_activity,
    MIN(date_acquired) AS earliest_activity,
    MAX(date_acquired) AS latest_activity,
    COUNT(*) AS total_activity_events
FROM steam.history
WHERE date_acquired IS NOT NULL;
"""

full_agregation_sample = pd.read_sql(query, engine)

full_agregation_sample.head()

,players_with_activity,earliest_activity,latest_activity,total_activity_events
0,45459,1970-01-01 05:00:00,2025-01-08 06:13:16,125250422


In [10]:
full_agregation_sample.to_csv(
    PROCESSED_DATA_PATH / "full_agregation_sample.csv",
    index=False
)

## Activity Date Validation

The first aggregation revealed suspicious activity dates starting from 1970-01-01.
Such values are likely default or invalid timestamps and would distort lifecycle metrics such as first activity date and active days.

Before creating the final player-level analytical dataset, we validate the distribution of activity dates and decide whether invalid historical dates should be excluded.

In [12]:
query = """
SELECT
    COUNT(*) AS records_with_1970_date,
    COUNT(DISTINCT player_id) AS players_with_1970_date
FROM steam.history
WHERE date_acquired < DATE '2000-01-01';
"""

sus_sample = pd.read_sql(query, engine)

sus_sample.head()

,records_with_1970_date,players_with_1970_date
0,92416,5333


In [13]:
sus_sample.to_csv(
    PROCESSED_DATA_PATH / "sus_sample.csv",
    index=False
)

In [14]:
query = """
SELECT
    EXTRACT(YEAR FROM date_acquired)::int AS activity_year,
    COUNT(*) AS activity_events,
    COUNT(DISTINCT player_id) AS players
FROM steam.history
WHERE date_acquired IS NOT NULL
GROUP BY EXTRACT(YEAR FROM date_acquired)::int
ORDER BY activity_year;
"""

date_distribution = pd.read_sql(query, engine)

date_distribution.head()

,activity_year,activity_events,players
0,1970,92416,5333
1,2008,290,62
2,2009,29809,1713
3,2010,169466,3295
4,2011,347433,4575


In [15]:
date_distribution.to_csv(
    PROCESSED_DATA_PATH / "date_distribution.csv",
    index=False
)

In [16]:
query = """
SELECT
    MIN(date_acquired) AS earliest_valid_activity,
    MAX(date_acquired) AS latest_valid_activity,
    COUNT(*) AS valid_activity_events,
    COUNT(DISTINCT player_id) AS players_with_valid_activity
FROM steam.history
WHERE date_acquired >= DATE '2000-01-01';
"""

early_activity = pd.read_sql(query, engine)

early_activity.head()

,earliest_valid_activity,latest_valid_activity,valid_activity_events,players_with_valid_activity
0,2008-09-12 05:18:17,2025-01-08 06:13:16,125158006,45455


In [17]:
early_activity.to_csv(
    PROCESSED_DATA_PATH / "early_activity.csv",
    index=False
)

## Data Quality Decision: Invalid Activity Dates

The initial activity validation showed 92,416 records with dates before 2000, all falling into 1970.
These records likely represent invalid default timestamps rather than real Steam activity.

Because the first realistic activity date after excluding invalid timestamps is 2008-09-12, the analytical base will exclude activity records before 2008-01-01.

This is treated as a data quality rule. Later, additional cohort filters may be applied for modeling or comparison purposes.

In [18]:
query = """
SELECT
    MIN(date_acquired) AS earliest_valid_activity,
    MAX(date_acquired) AS latest_valid_activity,
    COUNT(*) AS valid_activity_events,
    COUNT(DISTINCT player_id) AS players_with_valid_activity
FROM steam.history
WHERE date_acquired >= DATE '2000-01-01';
"""

realistic_activity = pd.read_sql(query, engine)

realistic_activity.head()

,earliest_valid_activity,latest_valid_activity,valid_activity_events,players_with_valid_activity
0,2008-09-12 05:18:17,2025-01-08 06:13:16,125158006,45455


In [19]:
early_activity.to_csv(
    PROCESSED_DATA_PATH / "realistic_activity.csv",
    index=False
)

In [20]:
query = """
WITH valid_history AS (
    SELECT
        player_id,
        achievement_id,
        date_acquired
    FROM steam.history
    WHERE date_acquired >= DATE '2008-01-01'
),

player_activity AS (
    SELECT
        vh.player_id,
        MIN(vh.date_acquired) AS first_activity_date,
        MAX(vh.date_acquired) AS last_activity_date,
        COUNT(*) AS total_achievements_unlocked,
        COUNT(DISTINCT a.game_id) AS games_with_activity,
        DATE_PART('day', MAX(vh.date_acquired) - MIN(vh.date_acquired)) AS active_days
    FROM valid_history vh
    LEFT JOIN steam.achievements a
        ON vh.achievement_id = a.achievement_id
    GROUP BY vh.player_id
)

SELECT *
FROM player_activity
ORDER BY total_achievements_unlocked DESC
LIMIT 10;
"""

player_game_activity = pd.read_sql(query, engine)

player_game_activity.head()

,player_id,first_activity_date,last_activity_date,total_achievements_unlocked,games_with_activity,active_days
0,76561197970825215,2013-12-27 00:41:04,2024-12-12 20:04:00,1006552,14404,4003.0
1,76561198854461127,2018-08-19 09:56:05,2024-07-12 05:33:31,742208,4453,2153.0
2,76561198063115240,2013-07-26 22:32:28,2024-10-20 21:56:52,697052,3241,4103.0
3,76561198141387426,2014-08-10 07:52:05,2024-12-10 10:57:04,597965,2877,3775.0
4,76561197979667190,2009-01-09 21:14:11,2024-11-22 07:12:23,582934,2206,5795.0


In [21]:
player_game_activity.to_csv(
    PROCESSED_DATA_PATH / "player_game_activity.csv",
    index=False
)

In [22]:
query = """
WITH valid_history AS (
    SELECT
        player_id,
        achievement_id,
        date_acquired
    FROM steam.history
    WHERE date_acquired >= DATE '2008-01-01'
)

SELECT
    COUNT(*) AS total_valid_history_records,
    COUNT(a.achievement_id) AS matched_achievement_records,
    COUNT(*) - COUNT(a.achievement_id) AS unmatched_achievement_records
FROM valid_history vh
LEFT JOIN steam.achievements a
    ON vh.achievement_id = a.achievement_id;
"""

achievement_join_validation = pd.read_sql(query, engine)

achievement_join_validation.head()

,total_valid_history_records,matched_achievement_records,unmatched_achievement_records
0,125158006,125158006,0


In [23]:
achievement_join_validation.to_csv(
    PROCESSED_DATA_PATH / "achievement_join_validation.csv",
    index=False
)

In [24]:
query = """
WITH valid_history AS (
    SELECT
        player_id,
        achievement_id,
        date_acquired
    FROM steam.history
    WHERE date_acquired >= DATE '2008-01-01'
),

player_activity AS (
    SELECT
        vh.player_id,
        MIN(vh.date_acquired) AS first_activity_date,
        MAX(vh.date_acquired) AS last_activity_date,
        COUNT(*) AS total_achievements_unlocked,
        COUNT(DISTINCT a.game_id) AS games_with_activity,
        DATE_PART('day', MAX(vh.date_acquired) - MIN(vh.date_acquired)) AS active_days
    FROM valid_history vh
    LEFT JOIN steam.achievements a
        ON vh.achievement_id = a.achievement_id
    GROUP BY vh.player_id
),

purchases AS (
    SELECT
        player_id,
        CARDINALITY(library) AS games_owned
    FROM steam.purchased_games
),

friends_cnt AS (
    SELECT
        player_id,
        CARDINALITY(friends) AS friend_count
    FROM steam.friends
)

SELECT
    pa.player_id,
    pa.first_activity_date,
    pa.last_activity_date,
    pa.active_days,
    pa.total_achievements_unlocked,
    pa.games_with_activity,
    COALESCE(p.games_owned, 0) AS games_owned,
    COALESCE(f.friend_count, 0) AS friend_count
FROM player_activity pa
LEFT JOIN purchases p
    ON pa.player_id = p.player_id
LEFT JOIN friends_cnt f
    ON pa.player_id = f.player_id
ORDER BY pa.total_achievements_unlocked DESC
LIMIT 10;
"""

player_activity_ownership_friends = pd.read_sql(query, engine)

player_activity_ownership_friends.head()

,player_id,first_activity_date,last_activity_date,active_days,total_achievements_unlocked,games_with_activity,games_owned,friend_count
0,76561197970825215,2013-12-27 00:41:04,2024-12-12 20:04:00,4003.0,1006552,14404,25041,1039
1,76561198854461127,2018-08-19 09:56:05,2024-07-12 05:33:31,2153.0,742208,4453,4379,1532
2,76561198063115240,2013-07-26 22:32:28,2024-10-20 21:56:52,4103.0,697052,3241,7097,1824
3,76561198141387426,2014-08-10 07:52:05,2024-12-10 10:57:04,3775.0,597965,2877,15685,1841
4,76561197979667190,2009-01-09 21:14:11,2024-11-22 07:12:23,5795.0,582934,2206,24170,0


In [25]:
player_activity_ownership_friends.to_csv(
    PROCESSED_DATA_PATH / "player_activity_ownership_friends.csv",
    index=False
)

In [28]:
query = """
SELECT
    COUNT(*) AS total_purchase_rows,
    COUNT(library) AS non_null_library_rows,
    COUNT(*) - COUNT(library) AS null_library_rows,
    MIN(CARDINALITY(library)) AS min_games_owned,
    MAX(CARDINALITY(library)) AS max_games_owned,
    ROUND(AVG(CARDINALITY(library))::numeric, 2) AS avg_games_owned
FROM steam.purchased_games;
"""

purchased_games_cardinality = pd.read_sql(query, engine)

purchased_games_cardinality.head()

,total_purchase_rows,non_null_library_rows,null_library_rows,min_games_owned,max_games_owned,avg_games_owned
0,102548,46941,55607,1,32463,239.85


In [29]:
purchased_games_cardinality.to_csv(
    PROCESSED_DATA_PATH / "purchased_games_cardinality.csv",
    index=False
)

In [26]:
query = """
SELECT
    COUNT(*) AS total_friend_rows,
    COUNT(friends) AS non_null_friend_rows,
    COUNT(*) - COUNT(friends) AS null_friend_rows,
    MIN(CARDINALITY(friends)) AS min_friend_count,
    MAX(CARDINALITY(friends)) AS max_friend_count,
    ROUND(AVG(CARDINALITY(friends))::numeric, 2) AS avg_friend_count
FROM steam.friends;
"""

friends_cardinality = pd.read_sql(query, engine)

friends_cardinality.head()

,total_friend_rows,non_null_friend_rows,null_friend_rows,min_friend_count,max_friend_count,avg_friend_count
0,424683,339461,85222,1,2000,91.04


In [27]:
friends_cardinality.to_csv(
    PROCESSED_DATA_PATH / "friends_cardinality.csv",
    index=False
)